# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rafayyk7/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

*   **Unit of Analysis (The Grain):** One row represents **one unique content page (URL)** belonging to a specific client.
*   **Time Window:** The feature lookback period utilizes 90 days of historical search performance data (Google Search Console metrics like impressions, clicks, CTR, and positions). The target label is evaluated over a subsequent 30-day forward-looking observation window.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import sys
import subprocess
import pandas as pd

# 1. Clone repository in Google Colab to access the data directory
if 'google.colab' in sys.modules:
    REPO_URL = "https://github.com/rafayyk7/flyrank-ml.git"
    REPO_DIR = "flyrank-ml"

    if not os.path.exists(REPO_DIR):
        print("Cloning repository into Google Colab...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)

    os.chdir(REPO_DIR)

# 2. Define data path
csv_path = 'data/raw/content_refresh_anonymized.csv'
if not os.path.exists(csv_path):
    csv_path = '../../data/raw/content_refresh_anonymized.csv'

# 3. Load dataset
df = pd.read_csv(csv_path)

# 4. Verify unique grain constraint
total_rows = len(df)
unique_combinations = df.groupby(['client_id', 'content_id']).ngroups

print("--- Unit of Analysis Verification ---")
print(f"Total rows in dataset: {total_rows:,}")
print(f"Unique client_id + content_id groups: {unique_combinations:,}")
if total_rows == unique_combinations:
    print("✅ Verified: One row = one unique content page per client. The grain is perfectly clean!")
else:
    print("❌ Duplicate grain detected! Rows contain duplicate page snapshots.")

Cloning repository into Google Colab...
--- Unit of Analysis Verification ---
Total rows in dataset: 30,000
Unique client_id + content_id groups: 30,000
✅ Verified: One row = one unique content page per client. The grain is perfectly clean!


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Here is the strict classification bucket contract for our columns in the Content Refresh and Decay lane:

1. **Features (Input Signals):**
   * `content_age_days`: Age of the content since creation (captures overall lifecycle).
   * `days_since_last_update`: Days elapsed since the content was last refreshed.
   * `impressions_90d`: Historical page visibility (marks traffic drop exposure).
   * `avg_position`: Mean organic ranking position in search results.
   * `ctr`: Click-Through Rate.
   * `word_count`: Total word count (checks for thin-content decay).

2. **Labels (Targets to Predict):**
   * `is_declining_label` (Binary target derived from `trend_direction == 'down'`).

3. **Context (Identifiers & Metadata):**
   * `client_id`: Used for client-holdout cross-validation splits.
   * `content_id`: Unique identifier for each page.

4. **Excluded (Leakage Protection):**
   * `trend_direction` & `trend_pct`: The source column and percentage metric used to construct the label. Including them would trigger catastrophic target leakage.
   * Any client-facing product flags (e.g., `health_score`, `needs_ctr_fix`): These are computed downstream of the decay event and would artificially inflate metrics during training.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Declare our structural data contract buckets
features = ["content_age_days", "days_since_last_update", "impressions_90d", "avg_position", "ctr", "word_count"]
label = "is_declining_label"
context = ["client_id", "content_id"]
excluded = ["trend_direction", "trend_pct"]

# Confirm no overlaps in the buckets
overlap_leakage = set(features).intersection(set(excluded))
print("--- Data Contract Bucket Audit ---")
if not overlap_leakage:
    print("✅ Verified: Target leakage variables are fully isolated and excluded from training features!")
else:
    print(f"❌ LEAKAGE WARNING: {overlap_leakage} is present in both features and excluded lists!")

--- Data Contract Bucket Audit ---
✅ Verified: Target leakage variables are fully isolated and excluded from training features!


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

This programmatic check runs our data contract audits:
1. **Target Distribution (Balance):** Assesses skewness between declining and stable pages.
2. **Missing Value Audit:** Identifies null counts within our feature columns to establish data-cleaning baseline rules.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Derive target label for verification
df['is_declining_label'] = df['trend_direction'].str.lower().eq('down').astype(int)

# 1. Target Balance Analysis
declining_count = df['is_declining_label'].sum()
stable_count = len(df) - declining_count
decay_rate = (declining_count / len(df)) * 100

print("--- Data Contract Quality Audit ---")
print(f"Target Label Split: Stable = {stable_count:,} | Declining = {declining_count:,}")
print(f"Base Decay Rate: {decay_rate:.2f}%")

# 2. Feature Null Audit
print("\n--- Feature Null Audit ---")
null_counts = df[features].isnull().sum()
for col, count in null_counts.items():
    print(f"Missing values in '{col}': {count:,} ({ (count/len(df))*100 :.2f}%)")

--- Data Contract Quality Audit ---
Target Label Split: Stable = 13,738 | Declining = 16,262
Base Decay Rate: 54.21%

--- Feature Null Audit ---
Missing values in 'content_age_days': 0 (0.00%)
Missing values in 'days_since_last_update': 0 (0.00%)
Missing values in 'impressions_90d': 0 (0.00%)
Missing values in 'avg_position': 0 (0.00%)
Missing values in 'ctr': 0 (0.00%)
Missing values in 'word_count': 7,699 (25.66%)


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

### What this data can NEVER tell us:
1. **Unbalanced Historical Snapshots:** The dataset is a structural snapshot of 32 clients. A model trained here might over-generalize seasonal fluctuations, underperforming when introduced to newly onboarded clients in vastly different sectors.
2. **The "Why" of Traffic Decay (User Intent):** While search metrics reveal trailing impression or click drops, they cannot tell us if the decay was caused by a competitor's pricing shift, a Google core algorithm update, or a natural drop in public consumer interest.
3. **Semantic/Content Quality Metrics:** We have access to numeric indicators (`word_count`) but zero access to the semantic quality, factual accuracy, readability, or readability score of the text.
4. **On-Page User Engagement:** The metrics represent external Google Search Console footprints only. We cannot observe what happens after a click—such as bounce rates, conversion rates, or on-page dwell times.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Data boundary limitations documented. Ready to save!")

Data boundary limitations documented. Ready to save!


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.